# Basket references

Exact correlated GBM transitions apply to both baskets. The geometric basket has normal log returns with variance $\tau(w\odot\sigma)^\top\rho(w\odot\sigma)$. The arithmetic basket is a sum of lognormals and is evaluated numerically.

## Geometric price and coordinate deltas

The five-asset case uses asymmetric spots, weights, dividends, volatilities and correlations. Its coordinate delta is the scalar geometric delta times $w_iG/S_i$.


In [ ]:
import sys
from pathlib import Path

import numpy as np
from IPython.display import Markdown, display

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))

from bsde_pricing.basket_reference import BasketReferencePricer
from bsde_pricing.experiment_design import ExperimentDesign
from bsde_pricing.geometric_basket import GeometricBasketReference
from bsde_pricing.monte_carlo import MonteCarloPricer
from bsde_pricing.problems import geometric_five_case

case = geometric_five_case()
exact = GeometricBasketReference(case.model, case.contract).evaluate(case.initial_spots)
mc = MonteCarloPricer(case.model, case.contract)
n_paths = 65_536
price = mc.estimate(case.initial_spots, n_paths=n_paths, rng=np.random.default_rng(2026))
lines = [
    f"Exact price **{exact.call_price:.6f}**. Monte Carlo **{price.price_estimate:.6f}** "
    f"(SE {price.standard_error:.6f}). Effective volatility **{exact.effective_volatility:.2%}**.\n",
    "| Asset | Exact delta | Pathwise delta | Pathwise SE | Paired bump |",
    "| ---: | ---: | ---: | ---: | ---: |",
]
for coordinate, delta in enumerate(exact.call_deltas):
    check = mc.estimate_coordinate_delta(case.initial_spots, coordinate=coordinate,
                                         n_paths=n_paths, rng=np.random.default_rng(2026))
    lines.append(f"| {coordinate+1} | {delta:.6f} | {check.pathwise.estimate:.6f} "
                 f"| {check.pathwise.standard_error:.6f} | {check.central_bump.estimate:.6f} |")
display(Markdown("\n".join(lines)))


Exact price **5.494719**. Monte Carlo **5.522089** (SE 0.034140). Effective volatility **14.12%**.

| Asset | Exact delta | Pathwise delta | Pathwise SE | Paired bump |
| ---: | ---: | ---: | ---: | ---: |
| 1 | 0.057186 | 0.057379 | 0.000237 | 0.057381 |
| 2 | 0.077201 | 0.077462 | 0.000320 | 0.077464 |
| 3 | 0.093577 | 0.093893 | 0.000388 | 0.093895 |
| 4 | 0.135441 | 0.135898 | 0.000562 | 0.135898 |
| 5 | 0.147050 | 0.147546 | 0.000610 | 0.147544 |


Each asset has a different weight and starting spot, which gives different coordinate deltas. All five pathwise estimates lie within one standard error of the exact values. Paired bumps agree to about $2\times10^{-6}$ on the same paths.

## Arithmetic prices with two sampling schemes

The arithmetic delta averages $e^{-rT}\mathbf1_{\{A_T>K\}}w_iS_T^i/S_0^i$. The geometric payoff and its pathwise delta contributions supply controls with known expectations. Independent pilot draws fix the coefficients. Both evaluation methods use 65,536 terminal states.


In [ ]:
case = ExperimentDesign.load().case("arithmetic-10")
pricer = BasketReferencePricer(case.model, case.contract)
anti = pricer.antithetic(case.initial_spots, n_pairs=32_768, seed=2027)
sobol = pricer.sobol(case.initial_spots, power=12, scrambles=16, seed=2028)
lines = ["| Method | Price | Plain SE | Controlled SE | Controlled 95% half-width |",
         "| --- | ---: | ---: | ---: | ---: |"]
for label, result in (("Antithetic", anti), ("Scrambled Sobol", sobol)):
    stat = result.controlled.price
    lines.append(f"| {label} | {stat.estimate:.6f} | {result.plain.price.standard_error:.6f} "
                 f"| {stat.standard_error:.6f} | {(stat.confidence_high-stat.confidence_low)/2:.6f} |")
display(Markdown("\n".join(lines)))


| Method | Price | Plain SE | Controlled SE | Controlled 95% half-width |
| --- | ---: | ---: | ---: | ---: |
| Antithetic | 6.406563 | 0.022702 | 0.002306 | 0.004519 |
| Scrambled Sobol | 6.405646 | 0.002383 | 0.001255 | 0.002674 |


The control reduces price uncertainty under both sampling schemes. Antithetic standard errors use independent pair averages. Sobol standard errors use 16 independent scramble means. The Sobol half-width of 0.00267 exceeds the locked reference budget of 0.002 at this demonstration allocation.

## Arithmetic coordinate deltas

Scaling by the asset weight puts the delta and its uncertainty in comparable units across dimensions.


In [ ]:
lines = ["| Asset | Sobol delta / weight | Sobol half-width / weight | Gap in combined SEs |",
         "| ---: | ---: | ---: | ---: |"]
for asset, (a, b, weight) in enumerate(zip(anti.controlled.deltas, sobol.controlled.deltas,
                                           case.contract.weights), start=1):
    half_width = (b.confidence_high-b.confidence_low)/(2*weight)
    gap = (b.estimate-a.estimate)/np.hypot(a.standard_error, b.standard_error)
    lines.append(f"| {asset} | {b.estimate/weight:.6f} | {half_width:.6f} | {gap:+.2f} |")
display(Markdown("\n".join(lines)))


| Asset | Sobol delta / weight | Sobol half-width / weight | Gap in combined SEs |
| ---: | ---: | ---: | ---: |
| 1 | 0.619735 | 0.001014 | +0.03 |
| 2 | 0.619725 | 0.001079 | -0.61 |
| 3 | 0.619749 | 0.000977 | -1.33 |
| 4 | 0.619642 | 0.001116 | -0.46 |
| 5 | 0.619574 | 0.000904 | -0.57 |
| 6 | 0.619627 | 0.001263 | -0.44 |
| 7 | 0.619705 | 0.001092 | -0.58 |
| 8 | 0.619791 | 0.001195 | -0.62 |
| 9 | 0.619525 | 0.001101 | -1.58 |
| 10 | 0.619280 | 0.001028 | -1.27 |


The true deltas agree by symmetry. The observed coordinate spread comes from sampling. The largest difference between methods is 1.58 combined standard errors. Their uncertainty statements are pointwise and the deltas share paths.
